# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [2]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Chandu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Chandu\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [3]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [4]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [5]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Use-Case Data!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [6]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [7]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [8]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [9]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 64, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [10]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node '10d18f'. Skipping!
Property 'summary' already exists in node '76ebef'. Skipping!
Property 'summary' already exists in node '610d46'. Skipping!
Property 'summary' already exists in node 'dcdd6a'. Skipping!
Property 'summary' already exists in node '733dd5'. Skipping!
Property 'summary' already exists in node 'ca35e4'. Skipping!
Property 'summary' already exists in node 'a50613'. Skipping!
Property 'summary' already exists in node '7b3ed0'. Skipping!
Property 'summary' already exists in node '28d3ec'. Skipping!
Property 'summary' already exists in node '35341a'. Skipping!
Property 'summary' already exists in node '23fcac'. Skipping!
Property 'summary' already exists in node 'c98eb8'. Skipping!
Property 'summary' already exists in node 'f0731d'. Skipping!
Property 'summary' already exists in node '55cc6d'. Skipping!
Property 'summary' already exists in node 'dd39d9'. Skipping!
Property 'summary' already exists in node '4774ed'. Skipping!
Property

Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/48 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '28d3ec'. Skipping!
Property 'summary_embedding' already exists in node 'ca35e4'. Skipping!
Property 'summary_embedding' already exists in node '76ebef'. Skipping!
Property 'summary_embedding' already exists in node 'dcdd6a'. Skipping!
Property 'summary_embedding' already exists in node '7b3ed0'. Skipping!
Property 'summary_embedding' already exists in node '10d18f'. Skipping!
Property 'summary_embedding' already exists in node '610d46'. Skipping!
Property 'summary_embedding' already exists in node '733dd5'. Skipping!
Property 'summary_embedding' already exists in node 'a50613'. Skipping!
Property 'summary_embedding' already exists in node '23fcac'. Skipping!
Property 'summary_embedding' already exists in node '35341a'. Skipping!
Property 'summary_embedding' already exists in node 'c98eb8'. Skipping!
Property 'summary_embedding' already exists in node 'f0731d'. Skipping!
Property 'summary_embedding' already exists in node 'dd39d9'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 86, relationships: 711)

We can save and load our knowledge graphs as follows.

In [ ]:
import json
import uuid
from datetime import datetime
from decimal import Decimal
from pathlib import Path
from ragas.testset.graph import KnowledgeGraph, Node, Relationship


def serialize(obj):
    """Recursively serialize objects to JSON-safe types."""
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    elif isinstance(obj, uuid.UUID):
        return str(obj)
    elif isinstance(obj, datetime):
        return obj.isoformat()
    elif isinstance(obj, Decimal):
        return float(obj)
    elif isinstance(obj, list):
        return [serialize(item) for item in obj]
    elif isinstance(obj, dict):
        return {str(k): serialize(v) for k, v in obj.items()}
    elif hasattr(obj, 'model_dump'):
        return serialize(obj.model_dump())
    elif hasattr(obj, '__dict__'):
        return serialize(vars(obj))
    elif hasattr(obj, '_asdict'):
        return serialize(obj._asdict())
    else:
        return str(obj)


def save_kg_robust(kg, filename):
    """Save KnowledgeGraph with full serialization and UTF-8 encoding."""
    try:
        data = {
            "nodes": [serialize(node) for node in kg.nodes],
            "relationships": [serialize(rel) for rel in kg.relationships]
        }
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2, sort_keys=True)
        print(f"✅ KnowledgeGraph saved successfully to {filename}")
        return True
    except Exception as e:
        print(f"❌ Error saving KnowledgeGraph: {e}")
        return False

# -------------------------------
# Load KnowledgeGraph safely
# -------------------------------
def load_kg_robust(filename):
    """Load KnowledgeGraph from JSON with UTF-8 decoding."""
    try:
        path = Path(filename)
        with open(path, "r", encoding="utf-8") as f:  # 👈 Force UTF-8
            data = json.load(f)

        nodes = [Node(**node_data) for node_data in data.get("nodes", [])]
        relationships = [Relationship(**rel_data) for rel_data in data.get("relationships", [])]

        kg = KnowledgeGraph(nodes=nodes, relationships=relationships)
        print(f"✅ KnowledgeGraph loaded successfully from {filename}")
        return kg
    except Exception as e:
        print(f"❌ Error loading KnowledgeGraph: {e}")
        return None


save_kg_robust(kg, "usecase_data_kg.json")

# Load
usecase_data_kg = load_kg_robust("usecase_data_kg.json")
print(usecase_data_kg)


✅ KnowledgeGraph saved successfully to usecase_data_kg.json
✅ KnowledgeGraph loaded successfully from usecase_data_kg.json
KnowledgeGraph(nodes: 86, relationships: 711)


Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [13]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [14]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

In [15]:
# Simplest fix - Use only one synthesizer
from ragas.testset.synthesizers import SingleHopSpecificQuerySynthesizer

# Create minimal query distribution
minimal_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 1.0)
]

# Generate with minimal setup
testset = generator.generate(
    testset_size=5,  # Start small
    query_distribution=minimal_distribution,
    raise_exceptions=False
)


Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/4 [00:00<?, ?it/s]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.


##### ✅ Answer:

##### SingleHopSpecificQuerySynthesizer (50%)

What it does: Creates simple, direct questions that can be answered from one piece of context.

Example: “When did the company start?”

Purpose: Tests whether your RAG system can find a single fact quickly.

##### MultiHopAbstractQuerySynthesizer (25%)

What it does: Creates broader, reasoning-style questions that need combining several parts of the document to form a general idea.

Example: “How has the company’s strategy changed over the years?”

Purpose: Tests whether your system can connect information and understand patterns or themes.

##### MultiHopSpecificQuerySynthesizer (25%)

What it does: Creates complex factual questions that need combining multiple specific pieces of data.

Example: “According to the 2021 and 2023 reports, what was the total number of new customers added?”

Purpose: Tests whether your system can combine details from different parts to get precise answers.

Finally, we can use our `TestSetGenerator` to generate our testset!

In [17]:
#testset = generator.generate(testset_size=10, query_distribution=query_distribution, raise_exceptions=False)
#testset.to_pandas()
df = testset.to_pandas()
print(f"✅ Generated {len(df)} test samples")
#print(df.head())

✅ Generated 4 test samples


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [18]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node 'ee2e44'. Skipping!
Property 'summary' already exists in node 'eed52b'. Skipping!
Property 'summary' already exists in node '01c229'. Skipping!
Property 'summary' already exists in node 'ad6f9c'. Skipping!
Property 'summary' already exists in node 'e4029a'. Skipping!
Property 'summary' already exists in node 'dda61b'. Skipping!
Property 'summary' already exists in node '7d73ee'. Skipping!
Property 'summary' already exists in node '5a2f0f'. Skipping!
Property 'summary' already exists in node 'd49671'. Skipping!
Property 'summary' already exists in node '4257a3'. Skipping!
Property 'summary' already exists in node '09514c'. Skipping!
Property 'summary' already exists in node '48108a'. Skipping!
Property 'summary' already exists in node 'd501bd'. Skipping!
Property 'summary' already exists in node 'bc42aa'. Skipping!
Property 'summary' already exists in node '6321e7'. Skipping!
Property 'summary' already exists in node 'd58928'. Skipping!
Property

...

Property 'summary_embedding' already exists in node '01c229'. Skipping!
Property 'summary_embedding' already exists in node 'eed52b'. Skipping!
Property 'summary_embedding' already exists in node 'ee2e44'. Skipping!
Property 'summary_embedding' already exists in node '5a2f0f'. Skipping!
Property 'summary_embedding' already exists in node '09514c'. Skipping!
Property 'summary_embedding' already exists in node 'dda61b'. Skipping!
Property 'summary_embedding' already exists in node 'ad6f9c'. Skipping!
Property 'summary_embedding' already exists in node 'e4029a'. Skipping!
Property 'summary_embedding' already exists in node 'd501bd'. Skipping!
Property 'summary_embedding' already exists in node '7d73ee'. Skipping!
Property 'summary_embedding' already exists in node '4257a3'. Skipping!
Property 'summary_embedding' already exists in node 'd49671'. Skipping!
Property 'summary_embedding' already exists in node '48108a'. Skipping!
Property 'summary_embedding' already exists in node 'aac78e'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [51]:
import pandas as pd

# Set display options to show all columns and full content
pd.set_option('display.max_columns', 10)  # Show all columns
pd.set_option('display.max_colwidth', 50)  # Show full content in each column
pd.set_option('display.width', 80)  # No width limit
pd.set_option('display.max_rows', 10)  # Show all rows

dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,"How does the study by Eloundou et al., 2025, c...",[Introduction ChatGPT launched in November 202...,"The study by Eloundou et al., 2025, is referen...",single_hop_specifc_query_synthesizer
1,What does the data indicate about the usage of...,[Table 1: ChatGPT daily message counts (millio...,"The data shows that in the US, ChatGPT message...",single_hop_specifc_query_synthesizer
2,Section what mean in ChatGPT use data?,[Variation by Occupation Figure 23 presents va...,The context discusses variation in ChatGPT usa...,single_hop_specifc_query_synthesizer
3,Wha is the month of November 2022?,[Conclusion This paper studies the rapid growt...,Conclusion This paper studies the rapid growth...,single_hop_specifc_query_synthesizer
4,How does the chatGPT usage for practical guida...,[<1-hop>\n\nConclusion This paper studies the ...,The context shows that the three most common c...,multi_hop_abstract_query_synthesizer
...,...,...,...,...
7,Whi ChatGPT usage varation by occpation and pr...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The data shows that ChatGPT usage varies signi...,multi_hop_abstract_query_synthesizer
8,"Based on the rapid growth of ChatGPT, which ha...",[<1-hop>\n\nConclusion This paper studies the ...,"The context indicates that by July 2025, ChatG...",multi_hop_specific_query_synthesizer
9,How many peaple used ChatGPT in 2025 and how d...,[<1-hop>\n\nConclusion This paper studies the ...,"By July 2025, ChatGPT had been used weekly by ...",multi_hop_specific_query_synthesizer
10,How does Handa et al. (2025) support the findi...,[<1-hop>\n\nTable 1: ChatGPT daily message cou...,Handa et al. (2025) provide detailed analysis ...,multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [22]:
from langsmith import Client

client = Client()

dataset_name = "Use Case1 Synthetic Data - AIE8"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [23]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [24]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [25]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [26]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [27]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG"
)

In [28]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [29]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [30]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [31]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [32]:
rag_chain.invoke({"question" : "What are people doing with AI these days?"})

'People are using AI in many different ways, both at work and outside of work. They employ generative AI to perform workplace tasks by either augmenting or automating human labor. AI is used for producing writing, software code, spreadsheets, and other digital products, which distinguishes it from traditional web search engines. Users interact with AI for tasks categorized broadly into Asking (seeking information or advice), Doing (producing output or accomplishing tasks), and Expressing (self-expression or personal reflection). Generative AI acts both as a co-worker producing outputs and as a co-pilot providing advice to improve human productivity. Additionally, AI usage includes therapy/companionship applications and engagement in games and role play, though these are less prevalent compared to workplace and productive uses.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [33]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [34]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

dopeness_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "dopeness": "Is this response dope, lit, cool, or is it just a generic response?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`:
- `labeled_helpfulness_evaluator`:
- `dopeness_evaluator`:

##### ✅ Answer:

##### qa_evaluator:
- Evaluates: Question-Answer Quality

- Uses LangSmith's built-in "qa" evaluator

- Measures how well the generated answer matches the expected answer

- Provides a general quality score for the Q&A pair

##### labeled_helpfulness_evaluator:
- Evaluates: Helpfulness of the Response

- Specifically checks if the submission is helpful to the user

- Compares the generated answer against the correct reference answer

- Uses the criteria: "Is this submission helpful to the user, taking into account the correct reference answer?"

- Takes into account both the prediction and the reference answer

##### dopeness_evaluator:
- Evaluates: Engagement and Style Quality

- Measures how "dope, lit, cool" the response is versus being generic

- Uses the criteria: "Is this response dope, lit, cool, or is it just a generic response?"

- Evaluates the creativity and engagement level of the response

- Focuses on whether the response is interesting and engaging rather than bland




## LangSmith Evaluation

In [35]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'perfect-collar-85' at:
https://smith.langchain.com/o/6b208d6f-f67e-4438-9319-6878a8bf7ac7/datasets/c7a95d1e-5489-4598-86ef-006b058405bb/compare?selectedSessions=d8a6a1b5-97d3-4fc1-ae46-88e9e6b2ad44




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,Based on the findings of Handa et al. (2025) a...,Based on the findings summarized in the provid...,None,Handa et al. (2025) report that nearly 80% of ...,1,1,0,7.153196,3829cdb2-ca5d-4562-a144-5ab2e29b0879,34d2386e-f233-4116-8d63-7f7dd8c2654d
1,How does Handa et al. (2025) support the findi...,Handa et al. (2025) is referenced as part of t...,None,Handa et al. (2025) provide detailed analysis ...,0,0,0,3.976536,0c4f48ce-e6ca-477c-bd29-bc260e066f11,ec4ab776-caed-49b9-bc23-4cfa9d01c26e
2,How many peaple used ChatGPT in 2025 and how d...,"By July 2025, ChatGPT had more than 700 millio...",None,"By July 2025, ChatGPT had been used weekly by ...",1,1,0,2.419209,b1d56282-0029-4291-b0f2-8a91ba5c5a7b,95224043-9920-43ce-967d-9bd0e0e845a0
3,"Based on the rapid growth of ChatGPT, which ha...","Based on the provided context, ChatGPT's rapid...",None,"The context indicates that by July 2025, ChatG...",1,1,0,7.336874,39aff9e5-d519-48c7-9725-86546e5f492b,936b2c02-0e8f-4c57-9e1b-0f949bf2f777
4,Whi ChatGPT usage varation by occpation and pr...,The data indicates that ChatGPT usage variatio...,None,The data shows that ChatGPT usage varies signi...,1,0,0,2.956870,29385568-5b54-4271-9ef5-9a6670e9670c,d8178991-54e0-4971-9f66-2f495c6b4c9a
5,Based on the data showing that non-work messag...,The growth in non-work messages to over 70% of...,None,The data indicates that non-work messages have...,1,1,0,6.427049,c1f3f9e3-3342-4a1f-ba66-3f44695e4ffc,d275c567-72bc-4666-bd7e-238d69eb2e50
6,how ChatGPT use vary by job and work message s...,"Based on the provided context, ChatGPT usage v...",None,The context shows that ChatGPT usage varies a ...,1,1,0,9.110825,bb70576f-51b6-48c5-b13c-a68fe0fb5a75,da64d2db-bda1-45b5-b66a-d0e194ed58b4
7,How does the chatGPT usage for practical guida...,"Based on the provided context, Practical Guida...",None,The context shows that the three most common c...,1,1,0,7.990101,a293d79a-e2c4-4967-aabd-263a36af4db7,6ad79dac-2cb8-43bc-98da-a0db3f42225e
8,Wha is the month of November 2022?,"Based on the provided context, the month of No...",None,Conclusion This paper studies the rapid growth...,1,1,1,2.856271,cd93b956-baf3-413f-96fb-6d473a9349f3,06927d73-d6e8-4874-8a45-7363ffcb0fb4
9,Section what mean in ChatGPT use data?,I don't know.,None,The context discusses variation in ChatGPT usa...,1,0,0,1.630901,87c00abf-7205-4477-b5ea-00e7131b0562,7e74cef6-1cff-4bd5-a90c-94094f8e3102


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [36]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [37]:
rag_documents = docs

In [38]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

##### ✅ Answer:

Modifying the chunk size significantly impacts RAG application performance because it directly affects the quality and relevance of retrieved information. Here's why:

- Retrieval Quality: Chunk size determines how precisely information is retrieved , smaller chunks improve precision, larger ones capture more context.

- Embedding Accuracy: Very small chunks lose semantic meaning; very large ones add noise, reducing embedding quality.

- Search Efficiency: Smaller chunks increase the number of embeddings to search (slower); larger chunks reduce accuracy but speed up retrieval.

- Generation Quality: The LLM performs best when chunks contain enough but not excessive context.



In [39]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

##### ✅ Answer:

- Core of Retrieval: The embedding model determines how well queries and documents match semantically.

- Semantic Quality: Better models (like text-embedding-3-large) understand meaning and context more accurately.

- Retrieval Accuracy: High-quality embeddings lead to more relevant document chunks being retrieved.

- Performance Trade-off: Larger models give better results but cost more in compute and storage.

- End-to-End Impact: Weak embeddings cause poor retrieval, leading to lower-quality final answers.

In [40]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [41]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [42]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [45]:
dopeness_rag_chain.invoke({"question" : "How are people using AI to make money?"})

"Alright, let's crank this up to eleven! Based on the slick intel from the context, people are stacking cash with AI by treating it like a brainy advisor or research sidekick, not just a task robot. AI, especially ChatGPT, is boosting work mojo by handing out sharp decision support that cranks up productivity—especially in knowledge-heavy gigs where smart calls mean serious $$$. So instead of just automating grunt work, AI is leveling up human smarts to make work faster, sharper, and ultimately more profitable. Collis and Brynjolfsson even lock in a mind-blowing $97 billion a year consumer surplus vibe in the US from this AI wizardry. Boom—AI ain’t just replacing work, it’s turbocharging how humans make money by upgrading their decision game and research hustle!"

Finally, we can evaluate the new chain on the same test set!

In [46]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'crazy-spring-26' at:
https://smith.langchain.com/o/6b208d6f-f67e-4438-9319-6878a8bf7ac7/datasets/c7a95d1e-5489-4598-86ef-006b058405bb/compare?selectedSessions=66cdea91-db5b-4373-a27a-e8b541a3ab49




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,Based on the findings of Handa et al. (2025) a...,"Oh snap, let's break this down with some turbo...",None,Handa et al. (2025) report that nearly 80% of ...,1,1,1,8.570937,3829cdb2-ca5d-4562-a144-5ab2e29b0879,042f1248-2e68-4b3d-82ce-221d97bafdac
1,How does Handa et al. (2025) support the findi...,"Alright, buckle up for this slick breakdown: H...",None,Handa et al. (2025) provide detailed analysis ...,1,0,1,5.730832,0c4f48ce-e6ca-477c-bd29-bc260e066f11,b21c7da9-10b2-40d9-a78b-168700c287f1
2,How many peaple used ChatGPT in 2025 and how d...,"Yo, check this out — by the end of July 2025, ...",None,"By July 2025, ChatGPT had been used weekly by ...",1,1,1,4.297362,b1d56282-0029-4291-b0f2-8a91ba5c5a7b,35c049a0-ceb1-4e61-9cfe-7c7d8027df00
3,"Based on the rapid growth of ChatGPT, which ha...","Yo, buckle up because this ChatGPT juggernaut ...",None,"The context indicates that by July 2025, ChatG...",1,1,1,7.717682,39aff9e5-d519-48c7-9725-86546e5f492b,a4545365-a531-4da1-9791-7438ea958633
4,Whi ChatGPT usage varation by occpation and pr...,"Yo, the data’s serving a fresh take on ChatGPT...",None,The data shows that ChatGPT usage varies signi...,1,1,1,4.366153,29385568-5b54-4271-9ef5-9a6670e9670c,20938124-b2f5-4fcf-8e58-52faa566c151
5,Based on the data showing that non-work messag...,"Oh, buckle up — this AI usage data is pure 🔥! ...",None,The data indicates that non-work messages have...,1,1,1,7.780439,c1f3f9e3-3342-4a1f-ba66-3f44695e4ffc,2e0ae92e-3404-4750-8081-85a52847c338
6,how ChatGPT use vary by job and work message s...,"Yo, let’s break down the epic saga of how Chat...",None,The context shows that ChatGPT usage varies a ...,1,1,1,10.443022,bb70576f-51b6-48c5-b13c-a68fe0fb5a75,f3757de6-3f90-4e20-aeb6-e45ffd9283d6
7,How does the chatGPT usage for practical guida...,"Alright, buckle up for some next-level AI insi...",None,The context shows that the three most common c...,1,1,1,6.028488,a293d79a-e2c4-4967-aabd-263a36af4db7,7e48002a-b10b-4c77-baec-8a227a32f032
8,Wha is the month of November 2022?,November 2022 is the legendary kickoff month w...,None,Conclusion This paper studies the rapid growth...,1,1,1,2.563112,cd93b956-baf3-413f-96fb-6d473a9349f3,11d79b77-53dd-4a08-842e-b904892d4b2a
9,Section what mean in ChatGPT use data?,"Yo, diving into the vibe of *""Section""* in the...",None,The context discusses variation in ChatGPT usa...,0,1,1,5.531076,87c00abf-7205-4477-b5ea-00e7131b0562,1d63f0d8-9ec0-40f0-9762-cb462a6dfd97


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

##### ✅ Answer:

##### Baseline vs. Dopeness Chain Comparison

| Metric | Baseline Chain | Dopeness Chain | Change |
|--------|----------------|----------------|---------|
| **Correctness** | 0.8333 | 0.8333 | No change |
| **Dopeness** | 0.0833 | 1.00 | **+1,100%** |
| **Helpfulness** | 0.5833 | 0.8333 | **+43%** |
| **Latency** | 5.202s | 6.10s | +17% slower |
| **Cost** | $0.0202 | $0.013 | **-35%** |



- Dopeness skyrocketed due to a new engaging prompt style.

- Helpfulness improved from larger chunks and better embeddings.

- Correctness stayed the same, since both used the same model and knowledge base.

- Latency slightly increased as richer outputs and larger models took longer.

- Cost dropped significantly thanks to more efficient retrieval and fewer total chunks.

Dopeness Results:
![Dopeness Results](dopeness.png)

Baseline Results:
![Baseline Results](baseline.png)
